In [16]:
%pip install numpy pandas scipy scikit-learn plotly joblib torch umap-learn openpyxl
%pip install --upgrade kaleido nbformat

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
# ================================
# Record environment versions
# ================================

import sys
import platform
from importlib.metadata import version, PackageNotFoundError

packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "joblib",
    "torch",
    "umap-learn",
    "plotly",
    "openpyxl",
    "nbformat",
    "kaleido",
]

print("# ================================")
print("# Python environment")
print("# ================================")
print(f"python=={platform.python_version()}")
print(f"python_executable={sys.executable}")
print(f"platform={platform.platform()}")
print()

print("# ================================")
print("# Package versions")
print("# ================================")

lines = [
    "# ================================\n",
    "# Python environment\n",
    "# ================================\n",
    f"python=={platform.python_version()}\n",
    f"python_executable={sys.executable}\n",
    f"platform={platform.platform()}\n\n",
    "# ================================\n",
    "# Package versions\n",
    "# ================================\n",
]

for pkg in packages:
    try:
        pkg_version = version(pkg)
    except PackageNotFoundError:
        pkg_version = "not installed"

    print(f"{pkg}=={pkg_version}")
    lines.append(f"{pkg}=={pkg_version}\n")

with open("environment_versions.txt", "w", encoding="utf-8") as f:
    f.writelines(lines)

print("\nSaved environment versions to: environment_versions.txt")

# ================================
# Python environment
# ================================
python==3.13.3
python_executable=c:\Users\hrnbe\Greenbootcamps\Projects\Carbon&MOF\Paper 2\.venv\Scripts\python.exe
platform=Windows-11-10.0.26200-SP0

# ================================
# Package versions
# ================================
numpy==2.4.4
pandas==3.0.2
scipy==1.17.1
scikit-learn==1.8.0
joblib==1.5.3
torch==2.11.0
umap-learn==0.5.12
plotly==6.7.0
openpyxl==3.1.5
nbformat==5.10.4
kaleido==1.3.0

Saved environment versions to: environment_versions.txt


# Latent-Space Transfer Reliability Audit — Inference-Only Notebook

This notebook loads fixed pretrained artifacts and performs the inference-only reliability audit used for the revised iScience submission. It includes updated publication-ready figure export settings: larger canvases, bold/larger labels, simplified legends, high-resolution PNG export, and vector PDF/SVG export.

**No model training, retraining, fine-tuning, or hyperparameter optimization is performed here.**

## 0. Imports and global configuration

In [18]:
# =============================================================================
# 0. Imports and global configuration
# =============================================================================
import os
import re
import glob
import json
import random
import platform
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.neighbors import NearestNeighbors

try:
    from scipy.stats import spearmanr
except Exception:  # pragma: no cover
    spearmanr = None

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"
try:
    pio.renderers.default = "vscode"
except Exception:
    pio.renderers.default = "browser"


@dataclass(frozen=True)
class Config:
    seed: int = 42
    outdir: str = "models_disentangled"
    knn_k: int = 10
    edge_k: int = 5
    mc_samples: int = 40
    mc_batch: int = 512
    bins: int = 10
    min_pairs_per_bin: int = 12
    min_domain_pair_n: int = 20
    boot: int = 300
    alpha: float = 0.10
    fig_width: int = 1800
    fig_height: int = 1250
    fig_scale: int = 4


CFG = Config()

MODEL_DIR = os.path.join(CFG.outdir, "models")
SHAP_DIR = os.path.join(CFG.outdir, "shap")
SAVE_DIR = os.path.join(CFG.outdir, "results")
REVISION_DIR = os.path.join(SAVE_DIR, "reviewer_editor_revision_outputs")
VALIDATION_DIR = os.path.join(REVISION_DIR, "prediction_error_validation")
for _d in [SAVE_DIR, REVISION_DIR, VALIDATION_DIR]:
    os.makedirs(_d, exist_ok=True)

X_PATH = os.path.join(CFG.outdir, "X_input.npy")
DOMAINS_PATH = os.path.join(CFG.outdir, "domains.pkl")
Y_PATH = os.path.join(CFG.outdir, "Y_targets.npy")
T_PATH = os.path.join(CFG.outdir, "target_cols.pkl")
UNC_PATH = os.path.join(CFG.outdir, "predictions_ann_uncertainty.pkl")
SCALER_PATH = os.path.join(CFG.outdir, "feature_scaler.pkl")
FEAT_PATH = os.path.join(CFG.outdir, "feature_cols.pkl")

EXTRA_DATASETS = {
    "extra_carbon_exp": "extra_carbon_exp.xlsx",
    "extra_CTF_exp": "extra_CTF_exp.xlsx",
}
TRAIN_DOMAINS_FOR_OOD = ["carbon_exp", "mof_sim", "mof_exp"]
TARGETS_FOR_TABLES = ["Cg (F/g)", "Cv (F/cm^3)"]

DOMAIN_ORDER = ["carbon_exp", "mof_exp", "mof_sim", "extra_carbon_exp", "extra_CTF_exp", "CTF_exp"]
DOMAIN_COLORS = {
    "carbon_exp": "#1f77b4",
    "mof_exp": "#d62728",
    "mof_sim": "#2ca02c",
    "extra_carbon_exp": "#9467bd",
    "extra_CTF_exp": "#ff7f0e",
    "CTF_exp": "#9467bd",
}
DOMAIN_SYMBOLS = {
    "carbon_exp": "circle",
    "mof_exp": "triangle-up",
    "mof_sim": "square",
    "extra_carbon_exp": "diamond",
    "extra_CTF_exp": "cross",
    "CTF_exp": "diamond",
}
ZONE_COLORS = {"safe": "#2ca02c", "caution": "#ff7f0e", "unsafe": "#d62728"}

# =============================================================================
# Publication-ready figure settings
# =============================================================================
# Plotly uses pixel dimensions rather than true DPI metadata. The high-resolution
# PNG exports below are generated with large canvas sizes and scale=4, which is
# suitable for Word insertion and journal-quality raster output. Vector PDF/SVG
# exports are also written wherever possible.
PUB_FONT = "Arial"
PUB_FONT_BOLD = "Arial Black"
FIG_WIDTH = 1800
FIG_HEIGHT = 1250
FIG_HEIGHT_WIDE = 1050
FIG_SCALE = 4
TITLE_SIZE = 34
AXIS_TITLE_SIZE = 30
TICK_SIZE = 24
LEGEND_SIZE = 21
ANNOTATION_SIZE = 24
PANEL_LABEL_SIZE = 34
DEFAULT_MARKER_SIZE = 8
DENSE_MARKER_SIZE = 5
DEFAULT_OPACITY = 0.62
MAX_POINTS_PER_GROUP = 650

## 1. Helper functions

In [19]:
# =============================================================================
# 1. Helper functions
# =============================================================================
def set_reproducibility(seed: int = CFG.seed) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sanitize(s: object) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")


def category_orders_domain(domains: Iterable[str]) -> Dict[str, List[str]]:
    present = set(map(str, domains))
    ordered = [d for d in DOMAIN_ORDER if d in present]
    ordered += sorted([d for d in present if d not in ordered])
    return {"domain": ordered}


def normalize_target_name(t: object) -> str:
    s = str(t).strip()
    if s in {"Cg", "Cg_F_g", "Cg (F/g)", "gravimetric_capacitance_F_per_g"}:
        return "Cg (F/g)"
    if s in {"Cv", "Cv_F_cm3", "Cv (F/cm3)", "Cv (F/cm^3)", "volumetric_capacitance_F_per_cm3"}:
        return "Cv (F/cm^3)"
    return s


def target_pretty_name(t: object) -> str:
    return {
        "Cg (F/g)": "Gravimetric capacitance, Cg",
        "Cv (F/cm^3)": "Volumetric capacitance, Cv",
        "I_Ag": "Gravimetric current density",
        "I_e": "Ion-transport rate",
    }.get(str(t), str(t))


def target_difference_axis_label(t: object) -> str:
    return {
        "Cg (F/g)": "<b>Nearest-neighbor target-property discrepancy / F g<sup>−1</sup></b>",
        "Cv (F/cm^3)": "<b>Nearest-neighbor target-property discrepancy / F cm<sup>−3</sup></b>",
    }.get(str(t), "<b>Nearest-neighbor target-property discrepancy</b>")


def safe_spearman(x: Sequence[float], y: Sequence[float]) -> Tuple[float, float, int]:
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy(float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 5 or spearmanr is None:
        return np.nan, np.nan, int(mask.sum())
    rho, pval = spearmanr(x[mask], y[mask], nan_policy="omit")
    return float(rho), float(pval), int(mask.sum())


def zscore(v: Sequence[float]) -> np.ndarray:
    arr = np.asarray(v, float)
    mu = np.nanmean(arr)
    sd = np.nanstd(arr)
    if not np.isfinite(sd) or sd == 0:
        return np.zeros_like(arr, dtype=float)
    return (arr - mu) / (sd + 1e-12)


def safe_set_cliponaxis(fig: go.Figure, value: bool = True) -> go.Figure:
    """Safely set cliponaxis only for trace types that support it."""
    for tr in fig.data:
        try:
            if hasattr(tr, "cliponaxis"):
                tr.cliponaxis = value
        except Exception:
            pass
    return fig


def downsample_for_plot(
    df: pd.DataFrame,
    group_cols: Optional[List[str]] = None,
    max_points_per_group: int = MAX_POINTS_PER_GROUP,
    seed: int = CFG.seed,
) -> pd.DataFrame:
    """Downsample only for visualization to avoid overplotting.

    The underlying analyses and exported tables remain unchanged.
    """
    if df is None or df.empty:
        return df
    if group_cols is None or len(group_cols) == 0:
        if len(df) <= max_points_per_group:
            return df.copy()
        return df.sample(n=max_points_per_group, random_state=seed).copy()
    parts = []
    for _, g in df.groupby(group_cols, dropna=False):
        if len(g) > max_points_per_group:
            parts.append(g.sample(n=max_points_per_group, random_state=seed))
        else:
            parts.append(g)
    return pd.concat(parts, ignore_index=True)


def add_panel_label(
    fig: go.Figure,
    label: str,
    x: float = -0.075,
    y: float = 1.08,
    size: int = PANEL_LABEL_SIZE,
) -> go.Figure:
    """Add a bold panel label such as a, b, c, or d."""
    fig.add_annotation(
        x=x,
        y=y,
        xref="paper",
        yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(family=PUB_FONT_BOLD, size=size, color="black"),
        align="left",
    )
    return fig


def simplify_legend_names(fig: go.Figure) -> go.Figure:
    """Make legend labels reader-friendly and bold."""
    replacements = {
        "carbon_exp": "Carbon exp.",
        "mof_exp": "MOF exp.",
        "mof_sim": "MOF sim.",
        "extra_carbon_exp": "External carbon",
        "extra_CTF_exp": "External CTF",
        "CTF_exp": "CTF exp.",
        "safe": "Safe",
        "caution": "Caution",
        "unsafe": "Unsafe",
    }
    for tr in fig.data:
        try:
            name = str(tr.name)
            clean = replacements.get(name, name).replace("_", " ")
            if not clean.startswith("<b>"):
                tr.name = f"<b>{clean}</b>"
        except Exception:
            pass
    return fig


def apply_publication_style(
    fig: go.Figure,
    width: int = FIG_WIDTH,
    height: int = FIG_HEIGHT,
    title_size: int = TITLE_SIZE,
    axis_title_size: int = AXIS_TITLE_SIZE,
    tick_size: int = TICK_SIZE,
    legend_size: int = LEGEND_SIZE,
) -> go.Figure:
    """Apply large, manuscript-ready Plotly styling for Word/PDF insertion."""
    current_margin = fig.layout.margin.to_plotly_json() if fig.layout.margin else {}
    margin = dict(l=160, r=90, t=170, b=160)
    margin.update({k: v for k, v in current_margin.items() if v is not None})

    fig.update_layout(
        template="plotly_white",
        width=width,
        height=height,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family=PUB_FONT, size=tick_size, color="black"),
        title=dict(font=dict(family=PUB_FONT_BOLD, size=title_size, color="black"), x=0.02, xanchor="left"),
        legend=dict(
            font=dict(family=PUB_FONT_BOLD, size=legend_size, color="black"),
            title=dict(font=dict(family=PUB_FONT_BOLD, size=legend_size, color="black")),
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="rgba(0,0,0,0.35)",
            borderwidth=1,
            itemsizing="constant",
        ),
        margin=margin,
    )

    fig.update_xaxes(
        showline=True, linewidth=2.6, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2.0, ticklen=9, tickcolor="black",
        showgrid=True, gridcolor="rgba(0,0,0,0.11)", zeroline=False,
        title_font=dict(family=PUB_FONT_BOLD, size=axis_title_size, color="black"),
        tickfont=dict(family=PUB_FONT_BOLD, size=tick_size, color="black"),
        automargin=True,
    )
    fig.update_yaxes(
        showline=True, linewidth=2.6, linecolor="black", mirror=True,
        ticks="outside", tickwidth=2.0, ticklen=9, tickcolor="black",
        showgrid=True, gridcolor="rgba(0,0,0,0.11)", zeroline=False,
        title_font=dict(family=PUB_FONT_BOLD, size=axis_title_size, color="black"),
        tickfont=dict(family=PUB_FONT_BOLD, size=tick_size, color="black"),
        automargin=True,
    )

    for ann in fig.layout.annotations or []:
        txt = str(getattr(ann, "text", ""))
        txt = txt.replace("target=", "").replace("domain=", "").replace("_", " ")
        if not txt.startswith("<b>"):
            txt = f"<b>{txt}</b>"
        ann.text = txt
        ann.font = dict(family=PUB_FONT_BOLD, size=ANNOTATION_SIZE, color="black")

    try:
        fig.update_traces(
            marker=dict(size=DEFAULT_MARKER_SIZE, line=dict(width=0.7, color="rgba(0,0,0,0.55)")),
            selector=dict(mode="markers"),
        )
    except Exception:
        pass

    safe_set_cliponaxis(fig, value=False)
    simplify_legend_names(fig)
    return fig


def set_shared_axis_titles(
    fig: go.Figure,
    x_title: Optional[str] = None,
    y_title: Optional[str] = None,
    x_y: float = -0.16,
    y_x: float = -0.11,
    font_size: int = AXIS_TITLE_SIZE,
) -> go.Figure:
    """Use one shared x/y title for faceted figures."""
    if x_title is not None:
        fig.update_xaxes(title_text="")
        fig.add_annotation(
            x=0.5, y=x_y, xref="paper", yref="paper",
            text=x_title, showarrow=False,
            font=dict(family=PUB_FONT_BOLD, size=font_size, color="black"),
        )
    if y_title is not None:
        fig.update_yaxes(title_text="")
        fig.add_annotation(
            x=y_x, y=0.5, xref="paper", yref="paper",
            text=y_title, showarrow=False, textangle=-90,
            font=dict(family=PUB_FONT_BOLD, size=font_size, color="black"),
        )
    return fig


def save_plot(
    fig: go.Figure,
    stem: str,
    width: int = FIG_WIDTH,
    height: int = FIG_HEIGHT,
    show: bool = True,
) -> None:
    """Style, display, and export a Plotly figure in HTML, PNG, PDF, and SVG.

    PNG is exported at large pixel dimensions using FIG_SCALE. PDF/SVG are
    vector exports and should be preferred when assembling final figures.
    """
    apply_publication_style(fig, width=width, height=height)
    fig.write_html(stem + ".html", include_plotlyjs="cdn")

    for ext in ["pdf", "svg"]:
        try:
            fig.write_image(f"{stem}.{ext}", width=width, height=height, scale=1)
        except Exception as exc:
            print(f"⚠️ Could not export {stem}.{ext}: {exc}")

    try:
        fig.write_image(stem + ".png", width=width, height=height, scale=FIG_SCALE)
    except Exception as exc:
        print(f"⚠️ Could not export {stem}.png. Install/upgrade kaleido. Reason: {exc}")

    if show:
        try:
            fig.show(renderer="vscode")
        except Exception:
            try:
                fig.show(renderer="browser")
            except Exception:
                pass


def binned_edges(x: Sequence[float], bins: int = CFG.bins) -> Optional[np.ndarray]:
    """Return robust bin edges using the 1st–99th percentile range."""
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if finite.sum() < max(10, bins):
        return None
    lo, hi = np.nanpercentile(x[finite], [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(x[finite])), float(np.nanmax(x[finite]))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return None
    return np.linspace(lo, hi, int(bins) + 1)

def binned_q90_with_ci(
    x: Sequence[float],
    y: Sequence[float],
    bins: int = CFG.bins,
    min_per_bin: int = CFG.min_pairs_per_bin,
    boot: int = CFG.boot,
    alpha: float = CFG.alpha,
    seed: int = CFG.seed,
) -> pd.DataFrame:
    """Compute binned 90th-percentile discrepancy with bootstrap confidence intervals.

    Parameters
    ----------
    x : sequence
        Latent-neighbor distances.
    y : sequence
        Nearest-neighbor target-property discrepancies.
    bins : int
        Number of latent-distance bins.
    min_per_bin : int
        Minimum finite pairs required for a bin to be reported.
    boot : int
        Number of bootstrap resamples used for the confidence interval.
    alpha : float
        Two-sided interval width. alpha=0.10 gives a 90% CI.
    seed : int
        Random seed for bootstrap reproducibility.

    Returns
    -------
    pandas.DataFrame
        Columns: bin, x_mid, n, q90, q90_lo, q90_hi.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x, y = x[finite], y[finite]
    if len(x) < max(10, min_per_bin):
        return pd.DataFrame(columns=["bin", "x_mid", "n", "q90", "q90_lo", "q90_hi"])

    edges = binned_edges(x, bins=bins)
    if edges is None:
        return pd.DataFrame(columns=["bin", "x_mid", "n", "q90", "q90_lo", "q90_hi"])

    rng = np.random.default_rng(seed)
    bin_id = np.digitize(x, edges) - 1
    bin_id = np.clip(bin_id, 0, int(bins) - 1)

    rows = []
    for b in range(int(bins)):
        mask = bin_id == b
        if mask.sum() < int(min_per_bin):
            continue
        xb = x[mask]
        yb = y[mask]
        q90 = float(np.nanpercentile(yb, 90))

        if boot and boot > 1 and len(yb) > 1:
            boots = np.empty(int(boot), dtype=float)
            n = len(yb)
            for i in range(int(boot)):
                samp = rng.integers(0, n, size=n)
                boots[i] = np.nanpercentile(yb[samp], 90)
            q90_lo = float(np.nanpercentile(boots, 100 * (alpha / 2)))
            q90_hi = float(np.nanpercentile(boots, 100 * (1 - alpha / 2)))
        else:
            q90_lo = q90_hi = q90

        rows.append({
            "bin": int(b),
            "x_mid": float(np.nanmean(xb)),
            "n": int(mask.sum()),
            "q90": q90,
            "q90_lo": q90_lo,
            "q90_hi": q90_hi,
        })
    return pd.DataFrame(rows).sort_values("x_mid").reset_index(drop=True)

## 2. Model definition and loading

In [20]:
# =============================================================================
# 2. Model definition and loading
# =============================================================================
class Encoder(nn.Module):
    def __init__(self, in_dim: int, hidden: int, latent: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, latent)
        self.drop = nn.Dropout(dropout)

    def encode(self, x):
        h = self.drop(F.relu(self.fc1(x)))
        return self.drop(F.relu(self.fc2(h)))


def load_latent_encoder() -> Tuple[Encoder, str]:
    candidates = sorted(glob.glob(os.path.join(MODEL_DIR, "latent_encoder_*.pt")))
    if not candidates:
        raise FileNotFoundError(f"No latent_encoder_*.pt found in {MODEL_DIR}")
    ckpt_path = candidates[0]
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = ckpt["state_dict"]
    net = Encoder(
        in_dim=sd["fc1.weight"].shape[1],
        hidden=sd["fc1.weight"].shape[0],
        latent=sd["fc2.weight"].shape[0],
        dropout=ckpt.get("dropout", 0.2),
    )
    net.load_state_dict(sd, strict=False)
    net.eval()
    return net, os.path.basename(ckpt_path)


@torch.no_grad()
def encode_all_eval(encoder: Encoder, X_np: np.ndarray, batch: int = 1024) -> np.ndarray:
    encoder.eval()
    chunks = []
    for i in range(0, len(X_np), batch):
        xb = torch.tensor(X_np[i:i + batch]).float()
        chunks.append(encoder.encode(xb).numpy())
    return np.vstack(chunks)


def mc_dropout_latent_uncertainty(encoder: Encoder, X_np: np.ndarray, mc: int = CFG.mc_samples, batch: int = CFG.mc_batch) -> np.ndarray:
    encoder.train()
    samples = []
    with torch.no_grad():
        for s in range(mc):
            torch.manual_seed(CFG.seed + s)
            chunks = []
            for i in range(0, len(X_np), batch):
                xb = torch.tensor(X_np[i:i + batch]).float()
                chunks.append(encoder.encode(xb).numpy())
            samples.append(np.vstack(chunks))
    encoder.eval()
    Zs = np.stack(samples, axis=0)
    return Zs.std(axis=0).mean(axis=1).astype(float)

## 3. Core data loading and geometry

In [21]:
# =============================================================================
# 3. Core data loading and geometry
# =============================================================================
def load_required_data():
    for p in [X_PATH, DOMAINS_PATH]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing required artifact: {p}")
    X_train = np.load(X_PATH).astype(np.float32)
    domains_train = np.asarray(joblib.load(DOMAINS_PATH)).astype(str)
    if len(X_train) != len(domains_train):
        raise RuntimeError("X_input.npy and domains.pkl length mismatch.")

    has_targets = os.path.exists(Y_PATH) and os.path.exists(T_PATH)
    Y, targets = None, None
    if has_targets:
        Y = np.asarray(np.load(Y_PATH), float)
        targets = [normalize_target_name(t) for t in list(joblib.load(T_PATH))]
        if Y.shape[0] != X_train.shape[0]:
            print("⚠️ Y_targets row mismatch -> disabling target analyses.")
            Y, targets, has_targets = None, None, False
    return X_train, domains_train, Y, targets, has_targets


def load_or_compute_uncertainty(encoder: Encoder, X_train: np.ndarray) -> np.ndarray:
    if os.path.exists(UNC_PATH):
        try:
            pack = joblib.load(UNC_PATH)
            ann_mc = pack.get("ANN_MC", None)
            if ann_mc is not None and "std_epistemic" in ann_mc:
                std_epi = np.asarray(ann_mc["std_epistemic"], float)
                if std_epi.ndim == 2 and std_epi.shape[0] == len(X_train):
                    print("✅ Loaded epistemic uncertainty from predictions_ann_uncertainty.pkl")
                    return np.nanmean(std_epi, axis=1).astype(float)
        except Exception as exc:
            print("⚠️ Could not load uncertainty; computing MC-dropout latent uncertainty:", exc)
    print("⚠️ Computing MC-dropout latent uncertainty from encoder dropout.")
    U_epi = mc_dropout_latent_uncertainty(encoder, X_train)
    np.save(os.path.join(SAVE_DIR, "mc_dropout_latent_uncertainty.npy"), U_epi)
    return U_epi


def compute_core_metrics(Z_train: np.ndarray, domains_train: np.ndarray, U_epi: np.ndarray):
    centroid = np.nanmean(Z_train, axis=0)
    latent_radius = np.sqrt(np.sum((Z_train - centroid) ** 2, axis=1))

    k_use = int(max(3, min(CFG.knn_k, len(Z_train) - 1)))
    nbrs = NearestNeighbors(n_neighbors=k_use + 1).fit(Z_train)
    dist_all, idx_all = nbrs.kneighbors(Z_train)
    latent_distance = dist_all[:, 1:].mean(axis=1)
    latent_density = 1.0 / (latent_distance + 1e-12)

    dist_thresh = float(np.nanpercentile(latent_distance, 90))
    unc_thresh = float(np.nanpercentile(U_epi, 90))

    def _zone(d, u):
        if d <= dist_thresh and u <= unc_thresh:
            return "safe"
        if d > dist_thresh and u > unc_thresh:
            return "unsafe"
        return "caution"

    core_df = pd.DataFrame({
        "idx": np.arange(len(Z_train), dtype=int),
        "domain": domains_train,
        "latent_radius": latent_radius,
        "latent_distance": latent_distance,
        "latent_density": latent_density,
        "uncertainty_epistemic": U_epi,
    })
    core_df["zone"] = [_zone(d, u) for d, u in zip(latent_distance, U_epi)]
    core_df.to_csv(os.path.join(SAVE_DIR, "core_latent_metrics.csv"), index=False)
    return core_df, idx_all, dist_all, dist_thresh, unc_thresh

## 4. Core figures

In [22]:
# =============================================================================
# 4. Core figures
# =============================================================================
def make_core_figures(core_df: pd.DataFrame, dist_thresh: float, unc_thresh: float):
    """Generate clear standalone latent-geometry and safety-map figures.

    The previous crowded multi-panel figure is exported as separate panels so
    the final manuscript can insert the clearest panels at large size.
    """
    # Figure 1a: latent radius by domain
    fig = px.violin(
        core_df, x="domain", y="latent_radius", color="domain",
        color_discrete_map=DOMAIN_COLORS,
        category_orders=category_orders_domain(core_df["domain"]),
        box=True, points="outliers",
        title="<b>Latent distance from global centroid</b>",
    )
    add_panel_label(fig, "a")
    fig.update_layout(showlegend=False, xaxis_title="<b>Materials domain</b>", yaxis_title="<b>Latent distance from centroid</b>")
    save_plot(fig, os.path.join(SAVE_DIR, "Fig1a_latent_radius_by_domain"), width=1500, height=1100)

    # Figure 1b: local density by domain
    fig = px.violin(
        core_df, x="domain", y="latent_density", color="domain",
        color_discrete_map=DOMAIN_COLORS,
        category_orders=category_orders_domain(core_df["domain"]),
        box=True, points="outliers",
        title="<b>Local latent-space density</b>",
    )
    add_panel_label(fig, "b")
    fig.update_layout(showlegend=False, xaxis_title="<b>Materials domain</b>", yaxis_title="<b>Latent density</b>")
    save_plot(fig, os.path.join(SAVE_DIR, "Fig1b_latent_density_by_domain"), width=1500, height=1100)

    # Figure 1c: density versus radius
    plot_df = downsample_for_plot(core_df, group_cols=["domain"], max_points_per_group=MAX_POINTS_PER_GROUP)
    fig = px.scatter(
        plot_df, x="latent_radius", y="latent_density", color="domain", symbol="domain",
        color_discrete_map=DOMAIN_COLORS, symbol_map=DOMAIN_SYMBOLS,
        category_orders=category_orders_domain(plot_df["domain"]), opacity=DEFAULT_OPACITY,
        title="<b>Latent density versus distance from centroid</b>",
    )
    add_panel_label(fig, "c")
    fig.update_traces(marker=dict(size=9))
    fig.update_layout(xaxis_title="<b>Latent distance from centroid</b>", yaxis_title="<b>Latent density</b>", legend_title_text="<b>Domain</b>", legend=dict(orientation="v", x=1.02, y=1.0))
    save_plot(fig, os.path.join(SAVE_DIR, "Fig1c_latent_density_vs_radius"), width=1600, height=1150)

    # Figure 1d: epistemic uncertainty versus latent distance
    fig = px.scatter(
        plot_df, x="latent_distance", y="uncertainty_epistemic", color="domain", symbol="domain",
        color_discrete_map=DOMAIN_COLORS, symbol_map=DOMAIN_SYMBOLS,
        category_orders=category_orders_domain(plot_df["domain"]), opacity=DEFAULT_OPACITY,
        title="<b>Epistemic uncertainty versus latent-neighborhood distance</b>",
    )
    add_panel_label(fig, "d")
    fig.update_traces(marker=dict(size=9))
    fig.update_layout(xaxis_title="<b>Mean latent-neighbor distance</b>", yaxis_title="<b>Epistemic uncertainty</b>", legend_title_text="<b>Domain</b>", legend=dict(orientation="v", x=1.02, y=1.0))
    save_plot(fig, os.path.join(SAVE_DIR, "Fig1d_distance_vs_uncertainty"), width=1600, height=1150)

    # Figure 1e: internal training-domain safety map
    fig = px.scatter(
        plot_df, x="latent_distance", y="uncertainty_epistemic", color="zone", symbol="domain",
        color_discrete_map=ZONE_COLORS, symbol_map=DOMAIN_SYMBOLS, opacity=0.72,
        title="<b>Internal training-domain safety map</b>",
    )
    fig.add_vline(x=dist_thresh, line_dash="dash", line_color="black", line_width=3)
    fig.add_hline(y=unc_thresh, line_dash="dash", line_color="black", line_width=3)
    add_panel_label(fig, "e")
    fig.update_traces(marker=dict(size=9))
    fig.update_layout(xaxis_title="<b>Mean latent-neighbor distance</b>", yaxis_title="<b>Epistemic uncertainty</b>", legend_title_text="<b>Safety zone / domain</b>", legend=dict(orientation="v", x=1.02, y=1.0))
    save_plot(fig, os.path.join(SAVE_DIR, "Fig1e_internal_training_domain_safety_map"), width=1650, height=1200)

## 5. kNN edges and target-property discrepancy

In [23]:
# =============================================================================
# 5. kNN edges and target-property discrepancy
# =============================================================================
def build_knn_edges(Z_train: np.ndarray, domains_train: np.ndarray) -> pd.DataFrame:
    """Build directed kNN edges and export a clear standalone connectivity bar chart."""
    k_rep = int(max(2, min(CFG.edge_k, len(Z_train) - 1)))
    nn = NearestNeighbors(n_neighbors=k_rep + 1, metric="euclidean").fit(Z_train)
    drep, irep = nn.kneighbors(Z_train)
    rows = []
    for i in range(len(Z_train)):
        for rank in range(1, k_rep + 1):
            j = int(irep[i, rank])
            qd, nd = str(domains_train[i]), str(domains_train[j])
            rows.append({
                "query_idx": int(i), "neighbor_rank": int(rank), "neighbor_idx": j,
                "query_domain": qd, "neighbor_domain": nd, "domain_pair": f"{qd}→{nd}",
                "latent_distance": float(drep[i, rank]),
            })
    df_edges = pd.DataFrame(rows)
    df_edges.to_csv(os.path.join(SAVE_DIR, "latent_knn_neighbors.csv"), index=False)

    pair_counts = df_edges.groupby("domain_pair").size().reset_index(name="n_edges").sort_values("n_edges", ascending=False)
    pair_counts.to_csv(os.path.join(SAVE_DIR, "latent_domain_pair_edge_counts.csv"), index=False)
    pair_counts["domain_pair_pretty"] = (
        pair_counts["domain_pair"]
        .str.replace("carbon_exp", "Carbon exp.", regex=False)
        .str.replace("mof_exp", "MOF exp.", regex=False)
        .str.replace("mof_sim", "MOF sim.", regex=False)
        .str.replace("_", " ", regex=False)
    )

    fig = px.bar(pair_counts, x="domain_pair_pretty", y="n_edges", title="<b>Latent-neighborhood connectivity across domain pairs</b>")
    add_panel_label(fig, "a")
    fig.update_traces(
        text=pair_counts["n_edges"].astype(str), texttemplate="<b>%{text}</b>", textposition="outside",
        textfont=dict(family=PUB_FONT_BOLD, size=22, color="black"), marker_line_width=1.2,
        marker_line_color="black", cliponaxis=False,
    )
    fig.update_layout(showlegend=False, xaxis_title="<b>Query–neighbor domain pair</b>", yaxis_title="<b>No. nearest-neighbor edges</b>", margin=dict(l=150, r=70, t=170, b=260))
    fig.update_xaxes(tickangle=-30)
    save_plot(fig, os.path.join(SAVE_DIR, "Fig2a_domain_pair_knn_edges"), width=1750, height=1200)
    return df_edges

def compute_target_discrepancy(df_edges: pd.DataFrame, Y: np.ndarray, targets: List[str]) -> pd.DataFrame:
    rows = []
    for tj, tname in enumerate(targets):
        tname = normalize_target_name(tname)
        y = np.asarray(Y[:, tj], float)
        finite = np.isfinite(y)
        for e in df_edges.itertuples(index=False):
            qi, ni = int(e.query_idx), int(e.neighbor_idx)
            if finite[qi] and finite[ni]:
                rows.append({
                    "target": tname,
                    "query_idx": qi,
                    "neighbor_idx": ni,
                    "latent_distance": float(e.latent_distance),
                    "abs_error": float(abs(y[qi] - y[ni])),
                    "query_domain": str(e.query_domain),
                    "neighbor_domain": str(e.neighbor_domain),
                    "domain_pair": str(e.domain_pair),
                    "cross_domain": bool(e.query_domain != e.neighbor_domain),
                })
    df_transfer = pd.DataFrame(rows)
    df_transfer.to_csv(os.path.join(SAVE_DIR, "nearest_neighbor_target_property_discrepancy.csv"), index=False)
    return df_transfer


def make_local_uncertainty_proxy(Z_train: np.ndarray, domains_train: np.ndarray, Y: np.ndarray, targets: List[str], core_df: pd.DataFrame) -> pd.DataFrame:
    """Compute local target-property variability used for Figure S1 diagnostics."""
    k_use = int(max(3, min(CFG.knn_k, len(Z_train) - 1)))
    nn = NearestNeighbors(n_neighbors=k_use + 1, metric="euclidean").fit(Z_train)
    d_unc, i_unc = nn.kneighbors(Z_train)
    neigh_idx = i_unc[:, 1:]
    latent_radius_nn = d_unc[:, 1:].mean(axis=1)

    rows = []
    for tj, target in enumerate(targets):
        target = normalize_target_name(target)
        if target not in TARGETS_FOR_TABLES:
            continue
        y = np.asarray(Y[:, tj], float)
        for i in range(len(Z_train)):
            neigh_y = y[neigh_idx[i]]
            mask = np.isfinite(neigh_y)
            if mask.sum() < max(3, k_use // 2):
                continue
            rows.append({
                "target": target,
                "idx": int(i),
                "domain": str(domains_train[i]),
                "latent_radius": float(latent_radius_nn[i]),
                "latent_distance": float(core_df.loc[i, "latent_distance"]),
                "latent_density": float(core_df.loc[i, "latent_density"]),
                "local_std": float(np.nanstd(neigh_y[mask])),
            })
    df_unc = pd.DataFrame(rows)
    df_unc.to_csv(os.path.join(SAVE_DIR, "local_target_property_variability.csv"), index=False)
    return df_unc


def make_target_discrepancy_figures(df_transfer: pd.DataFrame, core_df: pd.DataFrame, df_unc: Optional[pd.DataFrame] = None):
    """Generate clearer target-discrepancy figures.

    The old crowded Figure 2-style output is split into standalone panels:
    Fig2a = kNN connectivity; Fig2b = discrepancy versus distance; Fig2c/d =
    target-specific discrepancy envelopes. Density and local-variability plots
    are exported to SI panels.
    """
    if df_transfer is None or df_transfer.empty:
        print("⚠️ No target-discrepancy data available.")
        return

    pair_n = df_transfer.groupby(["target", "domain_pair"]).size().reset_index(name="n")
    good_pairs = set(pair_n[pair_n["n"] >= CFG.min_domain_pair_n]["domain_pair"])
    df_plot = df_transfer[df_transfer["domain_pair"].isin(good_pairs)].copy() if good_pairs else df_transfer.copy()
    df_plot_vis = downsample_for_plot(df_plot, group_cols=["target", "domain_pair"], max_points_per_group=MAX_POINTS_PER_GROUP)

    # Figure 2b: target-property discrepancy versus distance
    fig = px.scatter(
        df_plot_vis, x="latent_distance", y="abs_error", color="domain_pair", facet_col="target",
        opacity=0.55, render_mode="svg",
        title="<b>Target-property discrepancy increases with latent-neighbor distance</b>",
        labels={"latent_distance": "Latent-neighbor distance", "abs_error": "Nearest-neighbor target-property discrepancy", "domain_pair": "Domain pair", "target": "Target property"},
    )
    add_panel_label(fig, "b", x=-0.075, y=1.13)
    fig.update_traces(marker=dict(size=DENSE_MARKER_SIZE))
    set_shared_axis_titles(fig, x_title="<b>Latent-neighbor distance</b>", y_title="<b>Nearest-neighbor target-property discrepancy</b>", x_y=-0.18, y_x=-0.105, font_size=AXIS_TITLE_SIZE)
    fig.update_layout(legend_title_text="<b>Domain pair</b>", legend=dict(orientation="v", x=1.02, y=1.0), margin=dict(l=180, r=390, t=190, b=185))
    save_plot(fig, os.path.join(SAVE_DIR, "Fig2b_target_discrepancy_vs_distance"), width=2100, height=1150)

    # Supplementary Figure S2a: discrepancy versus density
    density_map = core_df.set_index("idx")["latent_density"].to_dict()
    df_transfer = df_transfer.copy()
    df_transfer["query_density"] = df_transfer["query_idx"].map(density_map).astype(float)
    df_plot_density = df_transfer[df_transfer["domain_pair"].isin(good_pairs)].copy() if good_pairs else df_transfer.copy()
    df_plot_density = downsample_for_plot(df_plot_density, group_cols=["target", "domain_pair"], max_points_per_group=MAX_POINTS_PER_GROUP)

    fig = px.scatter(
        df_plot_density, x="query_density", y="abs_error", color="domain_pair", facet_col="target",
        opacity=0.55, render_mode="svg",
        title="<b>Target-property discrepancy concentrates in low-density regions</b>",
        labels={"query_density": "Latent density", "abs_error": "Nearest-neighbor target-property discrepancy", "domain_pair": "Domain pair", "target": "Target property"},
    )
    add_panel_label(fig, "a", x=-0.075, y=1.13)
    fig.update_traces(marker=dict(size=DENSE_MARKER_SIZE))
    fig.update_xaxes(matches=None)
    set_shared_axis_titles(fig, x_title="<b>Latent density</b>", y_title="<b>Nearest-neighbor target-property discrepancy</b>", x_y=-0.18, y_x=-0.105, font_size=AXIS_TITLE_SIZE)
    fig.update_layout(legend_title_text="<b>Domain pair</b>", legend=dict(orientation="v", x=1.02, y=1.0), margin=dict(l=180, r=390, t=190, b=185))
    save_plot(fig, os.path.join(SAVE_DIR, "FigS2a_discrepancy_vs_density"), width=2100, height=1150)

    # Binned q90 envelopes by target/domain-pair
    env_rows = []
    for (target, dp), g in df_transfer.groupby(["target", "domain_pair"]):
        if len(g) < CFG.min_domain_pair_n:
            continue
        env = binned_q90_with_ci(g["latent_distance"], g["abs_error"], seed=CFG.seed)
        if env.empty:
            continue
        env["target"] = target
        env["domain_pair"] = dp
        env_rows.append(env)
    env_df = pd.concat(env_rows, ignore_index=True) if env_rows else pd.DataFrame()
    env_df.to_csv(os.path.join(SAVE_DIR, "target_discrepancy_q90_envelopes.csv"), index=False)

    fig_name_by_target = {"Cg (F/g)": ("Fig2c_q90_discrepancy_envelope_Cg", "c"), "Cv (F/cm^3)": ("Fig2d_q90_discrepancy_envelope_Cv", "d")}
    for target in sorted(env_df["target"].unique()) if not env_df.empty else []:
        et = env_df[env_df["target"] == target].copy()
        fig = go.Figure()
        for dp in sorted(et["domain_pair"].unique()):
            g = et[et["domain_pair"] == dp].sort_values("x_mid")
            pretty_dp = dp.replace("carbon_exp", "Carbon exp.").replace("mof_exp", "MOF exp.").replace("mof_sim", "MOF sim.").replace("_", " ")
            fig.add_trace(go.Scatter(x=g["x_mid"], y=g["q90"], mode="lines+markers", name=f"<b>{pretty_dp}</b>", line=dict(width=4), marker=dict(size=12, line=dict(width=1.0, color="black"))))
            fig.add_trace(go.Scatter(x=np.r_[g["x_mid"], g["x_mid"][::-1]], y=np.r_[g["q90_hi"], g["q90_lo"][::-1]], fill="toself", line=dict(width=0), opacity=0.14, showlegend=False, hoverinfo="skip"))
        stem, panel_label = fig_name_by_target.get(target, (f"Fig2_q90_discrepancy_envelope_{sanitize(target)}", ""))
        if panel_label:
            add_panel_label(fig, panel_label)
        fig.update_layout(
            title=f"<b>Distance-binned target-property discrepancy envelope</b><br><b>{target_pretty_name(target)}</b>",
            xaxis_title="<b>Binned latent-neighbor distance</b>", yaxis_title=target_difference_axis_label(target),
            legend=dict(orientation="v", x=1.02, y=1.0, title="<b>Domain pair</b>", bgcolor="rgba(255,255,255,0.95)", bordercolor="rgba(0,0,0,0.35)", borderwidth=1),
            margin=dict(l=170, r=360, t=190, b=150),
        )
        save_plot(fig, os.path.join(SAVE_DIR, stem), width=1750, height=1200)

    # Figure 3a: median far-neighbor discrepancy by domain pair and target
    rel_rows = []
    for target in sorted(df_transfer["target"].unique()):
        dt = df_transfer[df_transfer["target"] == target].copy()
        for dp, g in dt.groupby("domain_pair"):
            if len(g) < CFG.min_domain_pair_n:
                continue
            q3 = np.nanpercentile(g["latent_distance"], 75)
            far = g[g["latent_distance"] >= q3]
            if far.empty:
                continue
            rel_rows.append({"target": target, "domain_pair": dp, "n_edges": int(len(g)), "median_far_neighbor_discrepancy": float(np.nanmedian(far["abs_error"])), "q3_distance": float(q3)})
    df_rel = pd.DataFrame(rel_rows)
    df_rel.to_csv(os.path.join(SAVE_DIR, "domain_pair_far_neighbor_reliability.csv"), index=False)
    if not df_rel.empty:
        df_rel["domain_pair_pretty"] = df_rel["domain_pair"].str.replace("carbon_exp", "Carbon exp.", regex=False).str.replace("mof_exp", "MOF exp.", regex=False).str.replace("mof_sim", "MOF sim.", regex=False).str.replace("_", " ", regex=False)
        fig = px.bar(df_rel, x="domain_pair_pretty", y="median_far_neighbor_discrepancy", color="target", barmode="group", title="<b>Domain-pair reliability from far-neighbor target-property discrepancy</b>")
        add_panel_label(fig, "a")
        fig.update_traces(texttemplate="<b>%{y:.1f}</b>", textposition="outside", textfont=dict(family=PUB_FONT_BOLD, size=22, color="black"), marker_line_width=1.2, marker_line_color="black", cliponaxis=False)
        fig.update_layout(xaxis_title="<b>Query–neighbor domain pair</b>", yaxis_title="<b>Median far-neighbor target-property discrepancy</b>", legend_title_text="<b>Target property</b>", legend=dict(orientation="v", x=1.02, y=1.0), margin=dict(l=170, r=330, t=190, b=230))
        fig.update_xaxes(tickangle=-25)
        save_plot(fig, os.path.join(SAVE_DIR, "Fig3a_domain_pair_far_neighbor_discrepancy"), width=1800, height=1200)

    # Supplementary local target-property variability figures
    if df_unc is not None and not df_unc.empty:
        df_unc_vis = downsample_for_plot(df_unc, group_cols=["target", "domain"], max_points_per_group=MAX_POINTS_PER_GROUP)
        fig = px.scatter(df_unc_vis, x="latent_radius", y="local_std", color="domain", symbol="domain", facet_col="target", color_discrete_map=DOMAIN_COLORS, symbol_map=DOMAIN_SYMBOLS, category_orders=category_orders_domain(df_unc_vis["domain"]), opacity=0.62, render_mode="svg", title="<b>Local target-property variability increases with latent radius</b>")
        add_panel_label(fig, "b", x=-0.075, y=1.13)
        fig.update_traces(marker=dict(size=DENSE_MARKER_SIZE))
        set_shared_axis_titles(fig, x_title="<b>Latent radius</b>", y_title="<b>Local target-property variability</b>", x_y=-0.18, y_x=-0.105, font_size=AXIS_TITLE_SIZE)
        fig.update_layout(legend_title_text="<b>Domain</b>", legend=dict(orientation="v", x=1.02, y=1.0), margin=dict(l=180, r=330, t=190, b=185))
        save_plot(fig, os.path.join(SAVE_DIR, "FigS2b_local_variability_vs_radius"), width=2100, height=1150)

        fig = px.scatter(df_unc_vis, x="latent_density", y="local_std", color="domain", symbol="domain", facet_col="target", color_discrete_map=DOMAIN_COLORS, symbol_map=DOMAIN_SYMBOLS, category_orders=category_orders_domain(df_unc_vis["domain"]), opacity=0.62, render_mode="svg", title="<b>Local target-property variability decreases in dense regions</b>")
        add_panel_label(fig, "c", x=-0.075, y=1.13)
        fig.update_traces(marker=dict(size=DENSE_MARKER_SIZE))
        set_shared_axis_titles(fig, x_title="<b>Latent density</b>", y_title="<b>Local target-property variability</b>", x_y=-0.18, y_x=-0.105, font_size=AXIS_TITLE_SIZE)
        fig.update_layout(legend_title_text="<b>Domain</b>", legend=dict(orientation="v", x=1.02, y=1.0), margin=dict(l=180, r=330, t=190, b=185))
        save_plot(fig, os.path.join(SAVE_DIR, "FigS2c_local_variability_vs_density"), width=2100, height=1150)

## 5b. Relaxed sensitivity analysis and representative-case maps

In [24]:
# =============================================================================
# 5b. Relaxed sensitivity analysis and representative-case maps
# =============================================================================
def make_relaxed_sensitivity_analysis(Z_train: np.ndarray, domains_train: np.ndarray, Y: np.ndarray, targets: List[str]):
    """Generate clearer Figure S1: relaxed kNN/binning sensitivity envelopes.

    Uses k = 10 transfer edges, 6 latent-distance bins, >=8 pairs per bin, and
    >=15 pairs per domain pair, matching the reviewer-response description.
    """
    relaxed_k = 10
    relaxed_bins = 6
    relaxed_min_bin = 8
    relaxed_min_pair = 15
    k = int(max(2, min(relaxed_k, len(Z_train) - 1)))
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(Z_train)
    dmat, imat = nn.kneighbors(Z_train)

    edge_rows = []
    for i in range(len(Z_train)):
        for rank in range(1, k + 1):
            j = int(imat[i, rank])
            qd, nd = str(domains_train[i]), str(domains_train[j])
            edge_rows.append({"query_idx": int(i), "neighbor_idx": j, "neighbor_rank": int(rank), "latent_distance": float(dmat[i, rank]), "query_domain": qd, "neighbor_domain": nd, "domain_pair": f"{qd}→{nd}"})
    df_edges_relaxed = pd.DataFrame(edge_rows)
    df_edges_relaxed.to_csv(os.path.join(SAVE_DIR, "relaxed_k10_latent_knn_neighbors.csv"), index=False)

    rows = []
    for tj, target in enumerate(targets):
        target = normalize_target_name(target)
        if target not in TARGETS_FOR_TABLES:
            continue
        y = np.asarray(Y[:, tj], float)
        finite = np.isfinite(y)
        for e in df_edges_relaxed.itertuples(index=False):
            qi, ni = int(e.query_idx), int(e.neighbor_idx)
            if finite[qi] and finite[ni]:
                rows.append({"target": target, "domain_pair": e.domain_pair, "latent_distance": float(e.latent_distance), "abs_error": float(abs(y[qi] - y[ni]))})
    df_relaxed = pd.DataFrame(rows)
    df_relaxed.to_csv(os.path.join(SAVE_DIR, "relaxed_k10_target_property_discrepancy.csv"), index=False)

    env_rows = []
    for (target, dp), g in df_relaxed.groupby(["target", "domain_pair"]):
        if len(g) < relaxed_min_pair:
            continue
        env = binned_q90_with_ci(g["latent_distance"], g["abs_error"], bins=relaxed_bins, min_per_bin=relaxed_min_bin, seed=CFG.seed + 1000)
        if env.empty:
            continue
        env["target"] = target
        env["domain_pair"] = dp
        env_rows.append(env)
    env_df = pd.concat(env_rows, ignore_index=True) if env_rows else pd.DataFrame()
    env_df.to_csv(os.path.join(SAVE_DIR, "FigS1_relaxed_sensitivity_envelopes.csv"), index=False)

    for target in sorted(env_df["target"].unique()) if not env_df.empty else []:
        et = env_df[env_df["target"] == target]
        fig = go.Figure()
        for dp in sorted(et["domain_pair"].unique()):
            g = et[et["domain_pair"] == dp].sort_values("x_mid")
            pretty_dp = dp.replace("carbon_exp", "Carbon exp.").replace("mof_exp", "MOF exp.").replace("mof_sim", "MOF sim.").replace("_", " ")
            fig.add_trace(go.Scatter(x=g["x_mid"], y=g["q90"], mode="lines+markers", name=f"<b>{pretty_dp}</b>", line=dict(width=4), marker=dict(size=12, line=dict(width=1.0, color="black"))))
            fig.add_trace(go.Scatter(x=np.r_[g["x_mid"], g["x_mid"][::-1]], y=np.r_[g["q90_hi"], g["q90_lo"][::-1]], fill="toself", line=dict(width=0), opacity=0.14, showlegend=False, hoverinfo="skip"))
        panel = "a" if target == "Cg (F/g)" else "b"
        add_panel_label(fig, panel)
        fig.update_layout(
            title=f"<b>Sensitivity of discrepancy envelopes to relaxed neighborhood thresholds</b><br><b>{target_pretty_name(target)}</b>",
            xaxis_title="<b>Binned latent-neighbor distance</b>", yaxis_title=target_difference_axis_label(target),
            legend=dict(orientation="v", x=1.02, y=1.0, title="<b>Domain pair</b>", bgcolor="rgba(255,255,255,0.95)", bordercolor="rgba(0,0,0,0.35)", borderwidth=1),
            margin=dict(l=170, r=360, t=190, b=150),
        )
        save_plot(fig, os.path.join(SAVE_DIR, f"FigS1_relaxed_sensitivity_{sanitize(target)}"), width=1750, height=1200)

def make_table2_case_maps(core_df: pd.DataFrame, table2: pd.DataFrame, dist_thresh: float, unc_thresh: float):
    """Generate representative reliability-case maps in distance–uncertainty space.

    The maps annotate the Table 2 cases directly on the safety map, matching the
    manuscript/SI example requested by the user.
    """
    if table2 is None or table2.empty:
        return

    # Use non-overlapping annotation offsets similar to the supplied example.
    offset_map = {
        "A": (-95, -15),
        "B": (-15, -55),
        "C": (-85, -35),
    }
    for target in [t for t in TARGETS_FOR_TABLES if t in set(table2["Target"])]:
        cases = table2[table2["Target"] == target].copy()
        if cases.empty:
            continue
        fig = px.scatter(
            core_df,
            x="latent_distance",
            y="uncertainty_epistemic",
            color="domain",
            symbol="domain",
            color_discrete_map=DOMAIN_COLORS,
            symbol_map=DOMAIN_SYMBOLS,
            category_orders=category_orders_domain(core_df["domain"]),
            opacity=0.72,
            title=f"<b>Representative Reliability Cases in Distance–Uncertainty Space</b><br><b>{target_pretty_name(target)}</b>",
        )
        fig.add_vline(x=dist_thresh, line_dash="dash", line_color="black", line_width=2)
        fig.add_hline(y=unc_thresh, line_dash="dash", line_color="black", line_width=2)

        for _, row in cases.iterrows():
            sample_id = int(row["Sample ID"])
            case_label = str(row["Case"])
            case_letter = case_label.split(":", 1)[0].strip()
            point = core_df.loc[core_df["idx"] == sample_id]
            if point.empty:
                continue
            x = float(point["latent_distance"].iloc[0])
            y = float(point["uncertainty_epistemic"].iloc[0])
            ax, ay = offset_map.get(case_letter, (-60, -40))
            fig.add_annotation(
                x=x,
                y=y,
                text=f"<b>{case_label}</b>",
                showarrow=True,
                arrowhead=2,
                arrowwidth=1.4,
                arrowcolor="rgba(0,0,0,0.85)",
                ax=ax,
                ay=ay,
                bgcolor="rgba(255,255,255,0.92)",
                bordercolor="black",
                borderwidth=1,
                font=dict(family="Arial Black", size=13, color="black"),
            )
        fig.update_layout(
            xaxis_title="<b>Mean latent distance</b>",
            yaxis_title="<b>Epistemic uncertainty</b>",
            legend_title_text="<b>Materials domain</b>",
        )
        save_plot(
            fig,
            os.path.join(REVISION_DIR, f"FigS_Table2_representative_cases_{sanitize(target)}"),
            width=1250,
            height=850,
        )

## 5c. Reviewer/editor audit tables and revision-ready text outputs

In [25]:
# =============================================================================
# 5c. Reviewer/editor audit tables and revision-ready text outputs
# =============================================================================
def write_reliability_indicator_audit(df_transfer: pd.DataFrame, core_df: pd.DataFrame) -> pd.DataFrame:
    """Reviewer-requested audit: reliability indicators vs local target-property discrepancy.

    This addresses Reviewer 2's concern about whether the proposed indicators
    relate to a measurable error-like quantity while preserving the manuscript's
    conservative wording: nearest-neighbor target-property discrepancy is a
    local representation-smoothness diagnostic, not direct supervised prediction
    error.
    """
    if df_transfer is None or df_transfer.empty:
        out = pd.DataFrame()
        out.to_csv(os.path.join(REVISION_DIR, "Table_reliability_indicators_vs_target_difference.csv"), index=False)
        return out

    work = df_transfer.copy()
    core_maps = core_df.set_index("idx")
    for source_col, new_col in [
        ("latent_density", "query_density"),
        ("uncertainty_epistemic", "query_uncertainty_epistemic"),
        ("latent_radius", "query_latent_radius"),
        ("zone", "query_zone"),
    ]:
        if source_col in core_maps.columns:
            work[new_col] = work["query_idx"].map(core_maps[source_col])

    indicator_specs = [
        ("latent_distance", "Mean latent-neighbor distance"),
        ("query_density", "Latent density"),
        ("query_uncertainty_epistemic", "Epistemic uncertainty"),
        ("query_latent_radius", "Latent radius"),
    ]
    rows = []
    for target, dt in work.groupby("target"):
        for col, label in indicator_specs:
            if col not in dt.columns:
                continue
            rho, pval, n = safe_spearman(dt[col], dt["abs_error"])
            if n < 10:
                continue
            rows.append({
                "target": normalize_target_name(target),
                "target_pretty": target_pretty_name(normalize_target_name(target)),
                "indicator": label,
                "indicator_column": col,
                "n_pairs": int(n),
                "spearman_rho_vs_nearest_neighbor_target_property_discrepancy": rho,
                "p_value": pval,
                "interpretation": (
                    "Association between the reliability indicator and nearest-neighbor "
                    "target-property discrepancy; this is a local representation-smoothness "
                    "diagnostic, not a replacement for external supervised prediction error."
                ),
            })
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(REVISION_DIR, "Table_reliability_indicators_vs_target_difference.csv"), index=False)
    return out


def write_revision_ready_texts() -> None:
    """Write conservative manuscript/SI text snippets and reviewer-response checklist.

    These outputs document how the final script addresses the reviewer/editor
    requests without inflating the scientific claims.
    """
    limitations = """
Limitations of the study

This study audits a fixed pretrained latent encoder and does not optimize, retrain, or compare alternative representation-learning architectures. The conclusions therefore describe the reliability structure of the learned representation used here, rather than establishing a universally optimal model for electrochemical energy-storage materials. The nearest-neighbor target-property discrepancy used in this work is a local representation-smoothness diagnostic, not a direct substitute for prospective experimental validation or supervised prediction error on entirely unseen materials. Accordingly, the framework identifies unsupported or high-risk deployment regimes; it does not guarantee predictive accuracy in supported regions without independent validation.

The study focuses on a limited set of porous materials domains, namely experimental porous carbons, experimental and simulated MOFs, and inference-only external carbon/CTF datasets, with primary emphasis on gravimetric and volumetric capacitance. Broader validation across additional materials classes, electrolytes, device configurations, and electrochemical targets will be required to establish generality. The safety thresholds used to define interpolation-like, cautionary, and high-risk extrapolative regions are empirical and based on the distribution of latent distance and epistemic uncertainty in the available data. These thresholds are interpretable and useful for deployment auditing, but they may require recalibration for other datasets, models, or screening objectives.
""".strip()

    fig_s3_text = """
Supplementary Figure S3 text

To assess whether the proposed reliability indicators are associated with predictive degradation, we performed an internal leave-one-out latent-neighborhood validation for samples with available gravimetric capacitance, Cg, and volumetric capacitance, Cv, labels. For each labeled sample, the target value was estimated from its nearest latent neighbors after excluding the query sample. The resulting absolute prediction error was compared with latent distance, local density, epistemic uncertainty, and nearest-neighbor target-property discrepancy.

This validation is intended as a supporting analysis rather than a replacement for prospective external testing. Nearest-neighbor target-property discrepancy is therefore interpreted as a local representation-smoothness diagnostic, not as direct prediction error. Directional associations between the leave-one-out prediction error and the reliability indicators support their use as conservative warning signals for deployment risk.

Figure S3. Internal validation of reliability indicators against leave-one-out latent-neighborhood prediction error. Absolute prediction error is compared with mean latent distance, latent density, epistemic uncertainty, and median nearest-neighbor target-property difference for gravimetric capacitance, Cg, and volumetric capacitance, Cv. Rows correspond to Cg and Cv, respectively. Dashed lines indicate linear trend guides, and Spearman correlation coefficients are shown in each panel. These indicators are interpreted as conservative deployment-risk diagnostics rather than direct substitutes for external supervised prediction-error estimates.
""".strip()

    reviewer_checklist = """
Reviewer/editor revision checklist implemented in this script

1. Dataset/domain sample counts and target availability are exported as Table_dataset_composition_by_domain.csv.
2. kNN values and rationales are exported as Table_kNN_parameter_rationale.csv.
3. Nearest-neighbor target-property discrepancy is used consistently as a local representation-smoothness diagnostic, not direct supervised prediction error.
4. Domain-pair transferability is audited through kNN connectivity, target-property discrepancy, binned q90 envelopes, and far-neighbor discrepancy summaries.
5. Representative case studies for Cg and Cv are exported as Table2_representative_reliability_cases_Cg_Cv.csv/xlsx and annotated on distance-uncertainty maps.
6. Internal leave-one-out latent-neighborhood validation against prediction error is exported as Figure S3 and its correlation table.
7. Relaxed supplementary sensitivity analysis is exported as Figure S2 using k=10, fewer bins, and relaxed pair thresholds.
8. External inference-only deployment/OOD diagnostics are exported as Figure 3f when external files and scaler/features are available.
9. Older calibrated P(safe) and risk-coverage figures are intentionally not generated because they are exploratory and not statistically stable enough for formal reporting.
10. All generated figures use bold manuscript-style axes, tick labels, legends, and annotations where supported by Plotly.
""".strip()

    with open(os.path.join(REVISION_DIR, "limitations_revision_text.md"), "w", encoding="utf-8") as f:
        f.write(limitations)
    with open(os.path.join(VALIDATION_DIR, "Figure_S3_internal_validation_text.md"), "w", encoding="utf-8") as f:
        f.write(fig_s3_text)
    with open(os.path.join(REVISION_DIR, "reviewer_editor_revision_checklist.md"), "w", encoding="utf-8") as f:
        f.write(reviewer_checklist)

## 6. Dataset/reproducibility tables and representative Table 2

In [26]:
# =============================================================================
# 6. Dataset/reproducibility tables and representative Table 2
# =============================================================================
def write_dataset_and_method_tables(core_df, Y, targets, has_targets):
    rows = []
    for dom in sorted(core_df["domain"].unique()):
        mask = core_df["domain"].eq(dom).to_numpy()
        row = {"domain": dom, "n_samples": int(mask.sum()), "fraction_of_dataset": float(mask.mean())}
        if has_targets:
            for tj, t in enumerate(targets):
                y = np.asarray(Y[:, tj], float)
                row[f"n_finite_{t}"] = int(np.isfinite(y[mask]).sum())
                row[f"fraction_finite_{t}"] = float(np.isfinite(y[mask]).mean())
        rows.append(row)
    df_comp = pd.DataFrame(rows)
    df_comp.to_csv(os.path.join(REVISION_DIR, "Table_dataset_composition_by_domain.csv"), index=False)

    df_knn = pd.DataFrame([
        {"analysis_step": "Local density and mean neighborhood distance", "k_value": CFG.knn_k,
         "rationale": "Moderately sized neighborhood reduces sensitivity to single outliers while preserving local geometry."},
        {"analysis_step": "Domain-pair neighborhood connectivity and target-discrepancy edges", "k_value": CFG.edge_k,
         "rationale": "Smaller neighborhood emphasizes local cross-domain adjacency and avoids over-smoothing boundaries."},
        {"analysis_step": "Supplementary relaxed sensitivity analysis", "k_value": 10,
         "rationale": "Tests whether discrepancy-envelope trends remain stable under larger neighborhoods and relaxed bin-count thresholds."},
    ])
    df_knn.to_csv(os.path.join(REVISION_DIR, "Table_kNN_parameter_rationale.csv"), index=False)

    manifest = {
        "python_version": platform.python_version(), "platform": platform.platform(), "seed": CFG.seed,
        "knn_k": CFG.knn_k, "edge_k": CFG.edge_k, "mc_samples": CFG.mc_samples,
        "bins": CFG.bins, "min_pairs_per_bin": CFG.min_pairs_per_bin, "min_domain_pair_n": CFG.min_domain_pair_n,
        "numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__,
    }
    with open(os.path.join(REVISION_DIR, "software_and_reproducibility_manifest.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


def build_table2_cases(core_df: pd.DataFrame, df_transfer: pd.DataFrame):
    if df_transfer.empty:
        return pd.DataFrame()
    all_tables = []
    for target in [t for t in TARGETS_FOR_TABLES if t in set(df_transfer["target"])] :
        dt = df_transfer[df_transfer["target"] == target].copy()
        per_sample = dt.groupby("query_idx", as_index=False).agg(
            median_neighbor_target_difference=("abs_error", "median"),
            q90_neighbor_target_difference=("abs_error", lambda x: np.nanpercentile(x, 90)),
            mean_neighbor_latent_distance=("latent_distance", "mean"),
            n_neighbor_edges=("abs_error", "size"),
        )
        case_df = core_df.merge(per_sample, left_on="idx", right_on="query_idx", how="left")
        case_df["combined_risk_score"] = zscore(case_df["latent_distance"]) + zscore(case_df["uncertainty_epistemic"])
        q = {name: np.nanpercentile(case_df[col], pct) for name, col, pct in [
            ("density_hi", "latent_density", 75), ("density_lo", "latent_density", 25),
            ("err_lo", "median_neighbor_target_difference", 25), ("err_hi", "median_neighbor_target_difference", 75),
            ("ld_hi", "latent_distance", 75), ("unc_hi", "uncertainty_epistemic", 75),
        ]}
        selected = []

        def select(label, mask, sort_col, ascending=True, interpretation=""):
            sub = case_df[mask & np.isfinite(case_df[sort_col])].copy()
            if sub.empty:
                print(f"⚠️ No sample found for {target}: {label}")
                return
            row = sub.sort_values(sort_col, ascending=ascending).iloc[0].copy()
            row["Case"] = label
            row["Reliability interpretation"] = interpretation
            selected.append(row)

        select("A: Reliable interpolation", (case_df["zone"] == "safe") & (case_df["latent_density"] >= q["density_hi"]) & (case_df["median_neighbor_target_difference"] <= q["err_lo"]),
               "combined_risk_score", True, "Dense, low-distance neighborhood with low local target discontinuity")
        select("B: Silent-failure risk", (case_df["uncertainty_epistemic"] <= q["unc_hi"]) & (case_df["median_neighbor_target_difference"] >= q["err_hi"]),
               "median_neighbor_target_difference", False, "Low/moderate uncertainty but high local target discontinuity")
        select("C: Sparse extrapolative regime", (case_df["latent_distance"] >= q["ld_hi"]) & (case_df["latent_density"] <= q["density_lo"]),
               "latent_distance", False, "Sparse high-distance neighborhood; prediction should be rejected or treated cautiously")

        if not selected:
            continue
        out = pd.DataFrame(selected).drop_duplicates(subset=["idx"], keep="first")
        out["Target"] = target
        out = out.rename(columns={
            "idx": "Sample ID", "domain": "Domain", "zone": "Safety zone",
            "latent_distance": "Mean latent distance", "latent_density": "Latent density",
            "uncertainty_epistemic": "Epistemic uncertainty",
            "median_neighbor_target_difference": "Median nearest-neighbor target-property difference",
            "q90_neighbor_target_difference": "90th-percentile nearest-neighbor target-property difference",
            "n_neighbor_edges": "No. neighbor edges",
        })
        keep = ["Target", "Case", "Sample ID", "Domain", "Safety zone", "Mean latent distance", "Latent density",
                "Epistemic uncertainty", "Median nearest-neighbor target-property difference",
                "90th-percentile nearest-neighbor target-property difference", "No. neighbor edges", "Reliability interpretation"]
        all_tables.append(out[keep])
    table2 = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()
    if not table2.empty:
        num_cols = table2.select_dtypes(include=[np.number]).columns
        table2[num_cols] = table2[num_cols].round(3)
    table2.to_csv(os.path.join(REVISION_DIR, "Table2_representative_reliability_cases_Cg_Cv.csv"), index=False)
    table2.to_excel(os.path.join(REVISION_DIR, "Table2_representative_reliability_cases_Cg_Cv.xlsx"), index=False)
    return table2

## 7. Internal leave-one-out latent-neighborhood validation for Figure S3

In [27]:
# =============================================================================
# 7. Internal leave-one-out latent-neighborhood validation for Figure S3
# =============================================================================
def build_internal_validation(core_df: pd.DataFrame, df_edges: pd.DataFrame, Y: np.ndarray, targets: List[str], df_transfer: pd.DataFrame):
    rows = []
    target_to_col = {normalize_target_name(t): j for j, t in enumerate(targets)}
    for target in [t for t in TARGETS_FOR_TABLES if t in target_to_col]:
        y = np.asarray(Y[:, target_to_col[target]], float)
        for idx in range(len(y)):
            if not np.isfinite(y[idx]):
                continue
            neigh = df_edges[df_edges["query_idx"] == idx].sort_values("neighbor_rank")["neighbor_idx"].astype(int).tolist()
            vals = [y[j] for j in neigh if np.isfinite(y[j])]
            if len(vals) < 2:
                continue
            pred = float(np.nanmean(vals))
            rows.append({"idx": idx, "target": target, "y_true": float(y[idx]), "y_pred": pred,
                         "absolute_prediction_error": float(abs(y[idx] - pred))})
    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(os.path.join(VALIDATION_DIR, "loo_latent_neighbor_prediction_table_Cg_Cv.csv"), index=False)

    per_sample = df_transfer[df_transfer["target"].isin(TARGETS_FOR_TABLES)].groupby(["query_idx", "target"], as_index=False).agg(
        median_neighbor_target_difference=("abs_error", "median"),
        q90_neighbor_target_difference=("abs_error", lambda x: np.nanpercentile(x, 90)),
    ).rename(columns={"query_idx": "idx"})

    validation_df = pred_df.merge(core_df[["idx", "domain", "latent_distance", "latent_density", "uncertainty_epistemic", "zone"]], on="idx", how="left")
    validation_df = validation_df.merge(per_sample, on=["idx", "target"], how="left")
    validation_df.to_csv(os.path.join(VALIDATION_DIR, "prediction_error_reliability_validation_dataset_Cg_Cv.csv"), index=False)

    corr_rows = []
    indicators = [
        ("latent_distance", "Mean latent distance"),
        ("latent_density", "Latent density"),
        ("uncertainty_epistemic", "Epistemic uncertainty"),
        ("median_neighbor_target_difference", "Median nearest-neighbor target-property difference"),
    ]
    for target, dt in validation_df.groupby("target"):
        for col, label in indicators:
            rho, pval, n = safe_spearman(dt[col], dt["absolute_prediction_error"])
            corr_rows.append({"target": target, "indicator": label, "n_samples": n,
                              "spearman_rho_vs_absolute_prediction_error": rho, "p_value": pval})
    pd.DataFrame(corr_rows).to_csv(os.path.join(VALIDATION_DIR, "Table_prediction_error_vs_reliability_indicators_Cg_Cv.csv"), index=False)

    make_figS3_validation(validation_df)
    return validation_df


def make_figS3_validation(validation_df: pd.DataFrame):
    indicator_specs = [
        ("latent_distance", "<b>Mean latent<br>distance</b>"),
        ("latent_density", "<b>Latent<br>density</b>"),
        ("uncertainty_epistemic", "<b>Epistemic<br>uncertainty</b>"),
        ("median_neighbor_target_difference", "<b>Median NN target-property<br>difference</b>"),
    ]
    target_specs = [("Cg (F/g)", "Cg"), ("Cv (F/cm^3)", "Cv")]
    target_specs = [t for t in target_specs if t[0] in set(validation_df["target"])]
    n_rows, n_cols = len(target_specs), len(indicator_specs)
    if n_rows == 0:
        return

    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[lab if r == 0 else "" for r in range(n_rows) for _, lab in indicator_specs],
                        horizontal_spacing=0.095, vertical_spacing=0.195)
    domains = [d for d in DOMAIN_ORDER if d in set(validation_df["domain"])]
    domains += sorted([d for d in validation_df["domain"].dropna().unique() if d not in domains])
    shown = set()

    yvals = validation_df["absolute_prediction_error"].to_numpy(float)
    yvals = yvals[np.isfinite(yvals)]
    y_max = float(np.nanpercentile(yvals, 99.5) * 1.08) if len(yvals) else 1.0
    x_ranges = {}
    for col, _ in indicator_specs:
        x = pd.to_numeric(validation_df[col], errors="coerce").to_numpy(float)
        x = x[np.isfinite(x)]
        if len(x):
            lo, hi = float(np.nanmin(x)), float(np.nanpercentile(x, 99.5))
            pad = 0.06 * (hi - lo) if hi > lo else 0.1
            x_ranges[col] = [lo - pad, hi + pad]
        else:
            x_ranges[col] = [0, 1]

    for r, (target, _) in enumerate(target_specs, start=1):
        dt_target = validation_df[validation_df["target"] == target]
        for c, (x_col, _) in enumerate(indicator_specs, start=1):
            dt_panel = dt_target.dropna(subset=[x_col, "absolute_prediction_error"])
            for dom in domains:
                dd = dt_panel[dt_panel["domain"] == dom]
                if dd.empty:
                    continue
                showlegend = dom not in shown
                shown.add(dom)
                fig.add_trace(go.Scatter(
                    x=dd[x_col], y=dd["absolute_prediction_error"], mode="markers", name=f"<b>{dom}</b>",
                    legendgroup=dom, showlegend=showlegend,
                    marker=dict(size=7, color=DOMAIN_COLORS.get(dom, "#444444"), symbol=DOMAIN_SYMBOLS.get(dom, "circle"),
                                opacity=0.74, line=dict(width=0.35, color="rgba(0,0,0,0.30)")),
                    hovertemplate="Indicator=%{x:.3f}<br>Absolute error=%{y:.3f}<extra></extra>"), row=r, col=c)
            x = pd.to_numeric(dt_panel[x_col], errors="coerce").to_numpy(float)
            y = pd.to_numeric(dt_panel["absolute_prediction_error"], errors="coerce").to_numpy(float)
            m = np.isfinite(x) & np.isfinite(y)
            if m.sum() >= 5 and np.nanstd(x[m]) > 0:
                coef = np.polyfit(x[m], y[m], deg=1)
                xx = np.linspace(np.nanmin(x[m]), np.nanmax(x[m]), 100)
                fig.add_trace(go.Scatter(x=xx, y=coef[0] * xx + coef[1], mode="lines", showlegend=False,
                                         line=dict(color="rgba(80,80,80,0.80)", width=2, dash="dash"), hoverinfo="skip"), row=r, col=c)
            rho, _, n = safe_spearman(dt_panel[x_col], dt_panel["absolute_prediction_error"])
            panel = (r - 1) * n_cols + c
            xref = "x domain" if panel == 1 else f"x{panel} domain"
            yref = "y domain" if panel == 1 else f"y{panel} domain"
            if np.isfinite(rho):
                fig.add_annotation(x=0.055, y=0.925, xref=xref, yref=yref, text=f"<b>ρ = {rho:.2f}</b><br><b>n = {n}</b>",
                                   showarrow=False, align="left", bgcolor="rgba(255,255,255,0.84)",
                                   bordercolor="rgba(0,0,0,0.20)", borderwidth=1, font=dict(family="Arial Black", size=12, color="black"))
            fig.update_xaxes(range=x_ranges[x_col], title_text="", tickfont=dict(size=11), row=r, col=c)
            fig.update_yaxes(range=[0, y_max], title_text="", tickfont=dict(size=11), row=r, col=c)

    # Row labels and shared axes. Cg/Cv sit between the shared y-label and tick labels.
    if n_rows >= 1:
        fig.add_annotation(x=-0.085, y=0.735, xref="paper", yref="paper", text="<b>Cg</b>", showarrow=False,
                           textangle=-90, font=dict(family="Arial Black", size=18, color="black"))
    if n_rows >= 2:
        fig.add_annotation(x=-0.085, y=0.245, xref="paper", yref="paper", text="<b>Cv</b>", showarrow=False,
                           textangle=-90, font=dict(family="Arial Black", size=18, color="black"))
    fig.add_annotation(x=-0.165, y=0.500, xref="paper", yref="paper", text="<b>Absolute prediction error</b>",
                       showarrow=False, textangle=-90, font=dict(family="Arial Black", size=18, color="black"))
    fig.add_annotation(x=0.500, y=-0.145, xref="paper", yref="paper", text="<b>Reliability indicator value</b>",
                       showarrow=False, font=dict(family="Arial Black", size=16, color="black"))
    fig.update_layout(title=dict(text="<b>Internal Validation of Reliability Indicators</b><br><b>Against Leave-One-Out Latent-Neighborhood Prediction Error</b>",
                                 x=0.5, xanchor="center", y=0.990, yanchor="top", font=dict(size=23, color="black")),
                      width=2100, height=1080, plot_bgcolor="white", paper_bgcolor="white",
                      margin=dict(l=340, r=320, t=210, b=210),
                      legend=dict(title="<b>Materials domain</b>", x=1.065, y=0.960, xanchor="left", yanchor="top",
                                  font=dict(family="Arial Black", size=13, color="black"),
                                  title_font=dict(family="Arial Black", size=14, color="black"),
                                  bgcolor="rgba(255,255,255,0.98)", bordercolor="rgba(0,0,0,0.22)", borderwidth=1))
    fig.update_xaxes(showline=True, linewidth=1.4, linecolor="black", mirror=True, ticks="outside", ticklen=6, tickwidth=1.2,
                     showgrid=True, gridcolor="rgba(0,0,0,0.10)", zeroline=False,
                     tickfont=dict(family="Arial Black", size=11, color="black"))
    fig.update_yaxes(showline=True, linewidth=1.4, linecolor="black", mirror=True, ticks="outside", ticklen=6, tickwidth=1.2,
                     showgrid=True, gridcolor="rgba(0,0,0,0.10)", zeroline=False,
                     tickfont=dict(family="Arial Black", size=11, color="black"))
    stem = os.path.join(VALIDATION_DIR, "FigS3_prediction_error_validation_combined_Cg_Cv")
    fig.write_html(stem + ".html", include_plotlyjs="cdn")
    for ext in ["png", "pdf", "svg"]:
        try:
            fig.write_image(f"{stem}.{ext}", width=2100, height=1080, scale=3)
        except Exception as exc:
            if ext == "png":
                print("⚠️ Could not export Figure S3:", exc)
    try:
        fig.show(renderer="vscode")
    except Exception:
        fig.show(renderer="browser")

## 8. Optional external inference datasets and deployment map

In [28]:
# =============================================================================
# 8. Optional external inference datasets and deployment map
# =============================================================================
def run_extra_inference(encoder: Encoder, Z_train: np.ndarray, dist_thresh: float, unc_thresh: float):
    extra_tables = []
    if not (os.path.exists(SCALER_PATH) and os.path.exists(FEAT_PATH)):
        print("ℹ️ Scaler/features unavailable; skipping external inference datasets.")
        return pd.DataFrame()
    scaler = joblib.load(SCALER_PATH)
    features = list(joblib.load(FEAT_PATH))

    def build_X_extra(df_raw):
        Xr = df_raw.reindex(columns=features).apply(pd.to_numeric, errors="coerce").values
        mask = np.isfinite(Xr).astype(np.float32)
        Xs = scaler.transform(np.nan_to_num(Xr, nan=0.0)).astype(np.float32)
        return np.concatenate([Xs, mask], axis=1).astype(np.float32)

    nn = NearestNeighbors(n_neighbors=int(max(3, min(CFG.knn_k, len(Z_train) - 1))) + 1).fit(Z_train)
    for dom, fp in EXTRA_DATASETS.items():
        if not os.path.exists(fp):
            print(f"ℹ️ Extra dataset not found: {fp}")
            continue
        print(f"➕ Projecting external dataset: {fp} as {dom}")
        df = pd.read_excel(fp)
        Xx = build_X_extra(df)
        Zx = encode_all_eval(encoder, Xx)
        dx, _ = nn.kneighbors(Zx)
        Ux = mc_dropout_latent_uncertainty(encoder, Xx)
        out = pd.DataFrame({"domain": dom, "latent_distance": dx[:, 1:].mean(axis=1),
                            "uncertainty_epistemic": Ux, "source": "external"})
        out.to_csv(os.path.join(SAVE_DIR, f"external_{sanitize(dom)}_latent_metrics.csv"), index=False)
        extra_tables.append(out)
    return pd.concat(extra_tables, ignore_index=True) if extra_tables else pd.DataFrame()


def make_deployment_map(core_df: pd.DataFrame, extra_df: pd.DataFrame, dist_thresh: float, unc_thresh: float):
    train_map = core_df[["latent_distance", "uncertainty_epistemic", "domain"]].copy()
    train_map["source"] = "training"
    all_map = pd.concat([train_map, extra_df], ignore_index=True) if not extra_df.empty else train_map
    fig = px.scatter(all_map, x="latent_distance", y="uncertainty_epistemic", color="domain", symbol="source",
                     color_discrete_map=DOMAIN_COLORS, opacity=0.75,
                     title="<b>Deployment/OOD Map for Training and External Domains</b>")
    fig.add_vline(x=dist_thresh, line_dash="dash", line_color="black")
    fig.add_hline(y=unc_thresh, line_dash="dash", line_color="black")
    fig.update_layout(xaxis_title="<b>Mean latent distance</b>", yaxis_title="<b>Epistemic uncertainty</b>")
    save_plot(fig, os.path.join(SAVE_DIR, "Fig3f_deployment_OOD_map_training_plus_external"), width=1150, height=800)

## 9. SI summary tables

In [29]:
# =============================================================================
# 9. SI summary tables
# =============================================================================
def write_si_summary_tables(core_df: pd.DataFrame):
    df_safe = core_df.groupby("domain")["zone"].value_counts(normalize=True).unstack(fill_value=0).reset_index()
    for col in ["safe", "caution", "unsafe"]:
        if col not in df_safe:
            df_safe[col] = 0.0
    df_safe = df_safe.rename(columns={"safe": "safe_frac", "caution": "caution_frac", "unsafe": "unsafe_frac"})
    counts = core_df.groupby("domain").size().rename("n_samples").reset_index()
    df_safe = counts.merge(df_safe, on="domain", how="left")
    df_safe.to_csv(os.path.join(SAVE_DIR, "SI_Table_S1_safe_fraction_by_domain.csv"), index=False)

    train_mask = core_df["domain"].isin(TRAIN_DOMAINS_FOR_OOD)
    mu_d, sd_d = core_df.loc[train_mask, "latent_distance"].mean(), core_df.loc[train_mask, "latent_distance"].std()
    mu_u, sd_u = core_df.loc[train_mask, "uncertainty_epistemic"].mean(), core_df.loc[train_mask, "uncertainty_epistemic"].std()
    rows = []
    for dom, sub in core_df.groupby("domain"):
        if len(sub) < 3:
            continue
        rows.append({
            "domain": dom, "n_samples": int(len(sub)),
            "mean_latent_distance": float(sub["latent_distance"].mean()),
            "mean_uncertainty_epistemic": float(sub["uncertainty_epistemic"].mean()),
            "distance_shift_z": float((sub["latent_distance"].mean() - mu_d) / (sd_d + 1e-12)),
            "uncertainty_shift_z": float((sub["uncertainty_epistemic"].mean() - mu_u) / (sd_u + 1e-12)),
            "ood_flag": "train" if dom in TRAIN_DOMAINS_FOR_OOD else "OOD/external",
        })
    pd.DataFrame(rows).to_csv(os.path.join(SAVE_DIR, "SI_Table_S2_OOD_separation_score.csv"), index=False)

## 10. Main execution

In [30]:
# =============================================================================
# 10. Main execution
# =============================================================================
def main():
    set_reproducibility(CFG.seed)
    X_train, domains_train, Y, targets, has_targets = load_required_data()
    print("📦 Internal reference samples:", len(X_train))
    print("📦 Domain counts:", dict(zip(*np.unique(domains_train, return_counts=True))))
    print("🧪 Targets:", targets if has_targets else "not available")

    encoder, encoder_name = load_latent_encoder()
    print("🧠 Using latent encoder:", encoder_name)
    Z_train = encode_all_eval(encoder, X_train)
    U_epi = load_or_compute_uncertainty(encoder, X_train)
    core_df, idx_all, dist_all, dist_thresh, unc_thresh = compute_core_metrics(Z_train, domains_train, U_epi)

    make_core_figures(core_df, dist_thresh, unc_thresh)
    df_edges = build_knn_edges(Z_train, domains_train)
    write_dataset_and_method_tables(core_df, Y, targets, has_targets)

    if has_targets:
        df_transfer = compute_target_discrepancy(df_edges, Y, targets)
        df_unc = make_local_uncertainty_proxy(Z_train, domains_train, Y, targets, core_df)
        make_target_discrepancy_figures(df_transfer, core_df, df_unc=df_unc)
        write_reliability_indicator_audit(df_transfer, core_df)
        make_relaxed_sensitivity_analysis(Z_train, domains_train, Y, targets)
        table2 = build_table2_cases(core_df, df_transfer)
        make_table2_case_maps(core_df, table2, dist_thresh, unc_thresh)
        print("📄 Table 2 cases:")
        print(table2)
        build_internal_validation(core_df, df_edges, Y, targets, df_transfer)
    else:
        print("ℹ️ Target-dependent analyses skipped.")

    extra_df = run_extra_inference(encoder, Z_train, dist_thresh, unc_thresh)
    make_deployment_map(core_df, extra_df, dist_thresh, unc_thresh)
    write_si_summary_tables(core_df)
    write_revision_ready_texts()

    print("\n✅ FINAL PUBLISHABLE LATENT-RELIABILITY PIPELINE COMPLETE")
    print("📁 Outputs:", os.path.abspath(SAVE_DIR))
    print("📁 Reviewer/editor outputs:", os.path.abspath(REVISION_DIR))
    print("📁 Validation outputs:", os.path.abspath(VALIDATION_DIR))

if __name__ == "__main__":
    main()

📦 Internal reference samples: 242
📦 Domain counts: {np.str_('carbon_exp'): np.int64(122), np.str_('mof_exp'): np.int64(18), np.str_('mof_sim'): np.int64(102)}
🧪 Targets: ['Cg (F/g)', 'Cv (F/cm^3)']
🧠 Using latent encoder: latent_encoder_lambdaD0_05_lambdaN0_0.pt
⚠️ Computing MC-dropout latent uncertainty from encoder dropout.


📄 Table 2 cases:
        Target                            Case  Sample ID      Domain  \
0     Cg (F/g)       A: Reliable interpolation         62  carbon_exp   
1     Cg (F/g)          B: Silent-failure risk        191     mof_sim   
2     Cg (F/g)  C: Sparse extrapolative regime        190     mof_sim   
3  Cv (F/cm^3)       A: Reliable interpolation         40  carbon_exp   
4  Cv (F/cm^3)          B: Silent-failure risk        190     mof_sim   

  Safety zone  Mean latent distance  Latent density  Epistemic uncertainty  \
0        safe                 0.286           3.500                  0.937   
1     caution                 1.886           0.530                  0.948   
2     caution                 2.152           0.465                  0.901   
3        safe                 0.295           3.392                  0.846   
4     caution                 2.152           0.465                  0.901   

   Median nearest-neighbor target-property difference  \
0                 

➕ Projecting external dataset: extra_carbon_exp.xlsx as extra_carbon_exp
➕ Projecting external dataset: extra_CTF_exp.xlsx as extra_CTF_exp



✅ FINAL PUBLISHABLE LATENT-RELIABILITY PIPELINE COMPLETE
📁 Outputs: c:\Users\hrnbe\Greenbootcamps\Projects\Carbon&MOF\Paper 2\models_disentangled\results
📁 Reviewer/editor outputs: c:\Users\hrnbe\Greenbootcamps\Projects\Carbon&MOF\Paper 2\models_disentangled\results\reviewer_editor_revision_outputs
📁 Validation outputs: c:\Users\hrnbe\Greenbootcamps\Projects\Carbon&MOF\Paper 2\models_disentangled\results\reviewer_editor_revision_outputs\prediction_error_validation
